# 02. 응용 — 마라톤 fan-out, JoinNode, 결정론적 router

영상의 대표 예제를 API 키 없이 실행 가능한 ADK 2.8.0 그래프로 구현합니다.

## 학습 목표

1. 서로 독립적인 날씨·코스·체력 조회를 병렬로 fan-out한다.
2. JoinNode가 선행 노드 이름을 key로 결과를 묶는 방식을 확인한다.
3. 닫힌 선택지와 명시적 온도 신호에는 결정론적 router를 사용한다.
4. 선택된 전략 노드 하나만 실행되는지 검증한다.

전략 노드는 실제 LLM Agent 대신 deterministic stub입니다. model_calls=1은 실제
과금이 아니라, 해당 자리에 모델 Agent를 넣었을 때의 호출 예산을 나타냅니다.


In [ ]:
from importlib.metadata import version

assert version("google-adk") == "2.8.0", (
    "이 노트북은 google-adk==2.8.0으로 검증했습니다."
)


In [ ]:
import asyncio
import time

from google.adk import Event, Runner, Workflow
from google.adk.sessions import InMemorySessionService
from google.adk.workflow import DEFAULT_ROUTE, JoinNode
from google.genai import types


DELAYS = {"weather": 0.15, "course": 0.10, "fitness": 0.05}


async def fetch_weather(node_input):
    await asyncio.sleep(DELAYS["weather"])
    return Event(output={"temp_f": float(node_input.parts[0].text)})


async def fetch_course(node_input):
    await asyncio.sleep(DELAYS["course"])
    return Event(output={"elevation_m": 220, "surface": "road"})


async def fetch_fitness(node_input):
    await asyncio.sleep(DELAYS["fitness"])
    return Event(output={"weekly_km": 45, "long_run_km": 28})


join_inputs = JoinNode(name="join_inputs")


## 1. 경로를 코드로 고정하기

온도라는 기계 판독 가능한 값과 HOT/NORMAL/COLD라는 닫힌 집합이 있으므로 LLM
router가 필요하지 않습니다. 값이 없거나 숫자가 아니면 DEFAULT_ROUTE로 보내
조용히 종료되는 경로를 없앱니다.


In [ ]:
def route_by_temperature(node_input):
    try:
        temperature = float(node_input["fetch_weather"]["temp_f"])
    except (KeyError, TypeError, ValueError):
        return Event(output=node_input, route=DEFAULT_ROUTE)

    if temperature >= 80:
        route = "HOT"
    elif temperature <= 40:
        route = "COLD"
    else:
        route = "NORMAL"
    return Event(output=node_input, route=route)


def hot_strategy(node_input):
    return Event(
        output={
            "route": "HOT",
            "strategy": "초반 속도를 낮추고 급수 지점마다 수분을 보충한다.",
            "model_calls": 1,
        }
    )


def normal_strategy(node_input):
    return Event(
        output={
            "route": "NORMAL",
            "strategy": "목표 페이스를 균등하게 유지한다.",
            "model_calls": 1,
        }
    )


def cold_strategy(node_input):
    return Event(
        output={
            "route": "COLD",
            "strategy": "워밍업을 늘리고 체온 유지 장비를 준비한다.",
            "model_calls": 1,
        }
    )


def manual_review(node_input):
    return Event(
        output={
            "route": "MANUAL_REVIEW",
            "strategy": "온도 신호를 확인한 뒤 다시 실행한다.",
            "model_calls": 0,
        }
    )


## 2. fan-out → join → route 선언하기

첫 엣지의 tuple은 세 조회 노드를 동시에 시작합니다. 두 번째 엣지는 세 선행 노드가
모두 끝난 뒤 JoinNode와 router로 이동합니다. 마지막 dict는 route 값과 실행할
노드를 매핑합니다.


In [ ]:
race_workflow = Workflow(
    name="marathon_race_coach",
    edges=[
        ("START", (fetch_weather, fetch_course, fetch_fitness)),
        (
            (fetch_weather, fetch_course, fetch_fitness),
            join_inputs,
            route_by_temperature,
        ),
        (
            route_by_temperature,
            {
                "HOT": hot_strategy,
                "NORMAL": normal_strategy,
                "COLD": cold_strategy,
                DEFAULT_ROUTE: manual_review,
            },
        ),
    ],
)

[
    (edge.from_node.name, edge.to_node.name, edge.route)
    for edge in race_workflow.graph.edges
]


In [ ]:
async def run_race(temperature_f, session_suffix):
    service = InMemorySessionService()
    session = await service.create_session(
        app_name="graph_learning_lab",
        user_id="runner",
        session_id=f"practice-{session_suffix}",
    )
    runner = Runner(
        node=race_workflow,
        app_name="graph_learning_lab",
        session_service=service,
    )
    message = types.Content(
        role="user",
        parts=[types.Part(text=str(temperature_f))],
    )

    started = time.perf_counter()
    observed = []
    async for event in runner.run_async(
        user_id="runner",
        session_id=session.id,
        new_message=message,
    ):
        observed.append(
            {
                "output": event.output,
                "error_code": event.error_code,
            }
        )
    return observed, time.perf_counter() - started


hot_events, hot_elapsed = await run_race(86, "hot")
hot_events, round(hot_elapsed, 3)


## 3. 병렬성과 join 계약 검증하기

세 지연을 순차로 실행하면 약 0.30초가 필요합니다. 병렬 그래프는 가장 느린 분기
0.15초에 런타임 오버헤드가 더해진 수준이어야 합니다. 성능 assertion은 매우
느린 공유 환경에서도 흔들리지 않도록 여유를 둡니다.


In [ ]:
sequential_delay = sum(DELAYS.values())
joined = next(
    event["output"]
    for event in hot_events
    if isinstance(event["output"], dict)
    and set(event["output"]) == {
        "fetch_weather",
        "fetch_course",
        "fetch_fitness",
    }
)
final_hot = hot_events[-1]["output"]

assert hot_elapsed < sequential_delay * 0.9, (
    f"병렬 실행 {hot_elapsed:.3f}s가 순차 기준 {sequential_delay:.3f}s에 너무 가깝습니다."
)
assert joined["fetch_weather"]["temp_f"] == 86
assert joined["fetch_course"]["elevation_m"] == 220
assert joined["fetch_fitness"]["weekly_km"] == 45
assert final_hot["route"] == "HOT"
assert final_hot["model_calls"] == 1

{
    "병렬 실행 초": round(hot_elapsed, 3),
    "순차 지연 합계 초": sequential_delay,
    "join keys": sorted(joined),
    "최종 결과": final_hot,
}


JoinNode는 사전을 묶을 뿐 의미를 해석하거나 충돌을 해결하지 않습니다. 데이터
정규화, 우선순위, 누락 처리 등이 필요하면 별도 집계 함수 노드를 추가해야 합니다.
또한 실제 외부 API 비용과 CPU·네트워크 비용은 남아 있으므로 함수 노드의 zero
cost는 LLM 호출 비용이 0이라는 제한된 뜻입니다.


## 4. 모든 경로가 정확히 하나의 전략만 실행하는지 확인하기


In [ ]:
route_cases = [(86, "HOT"), (60, "NORMAL"), (35, "COLD")]
route_results = []

for index, (temperature, expected_route) in enumerate(route_cases):
    events, elapsed = await run_race(temperature, f"case-{index}")
    final = events[-1]["output"]
    assert final["route"] == expected_route
    assert final["model_calls"] == 1
    route_results.append(
        {
            "temperature_f": temperature,
            "expected": expected_route,
            "actual": final["route"],
            "simulated_model_calls": final["model_calls"],
        }
    )

route_results


## 핵심 정리와 확장 과제

- fan-out은 독립적인 I/O의 대기 시간을 합이 아니라 최댓값에 가깝게 줄입니다.
- JoinNode의 입력 key는 선행 노드 이름이므로 이름 변경도 데이터 계약 변경입니다.
- 닫힌 경로와 명시적 신호에는 코드 router가 저렴하고 재현 가능합니다.
- 이 데모의 모델 호출 1회는 그래프 일반 법칙이 아니라 이 노드 구성의 결과입니다.
- 실제 Agent로 바꾸면 모델 출력 schema, timeout, retry, 평가와 비용 한도가 필요합니다.

확장 과제: course와 fitness를 입력 schema로 검증하고, 한 분기가 실패했을 때
필수/선택 데이터 정책을 구분해 보세요. 다음 노트북에서 그 운영 정책을 구현합니다.
